# Reporting the M2 fine-tuning results

This notebook shows **which numbers to report** for the paper, following the
protocol in `research.org`.

Key rule: **`best_f1` computed on the test set is an oracle and must NOT be
reported.** The correct protocol is:

1. **plain `f1` (threshold 0.0)** = lower bound (conservative).
2. **dev-tuned threshold applied to test** = **headline**. The threshold is
   frozen from the dev set at the best epoch, then applied to the test
   predictions.
3. **`best_f1` on test** = never reported as a result (diagnostic only).

This notebook reads `metrics_summary.json`, `gold_audit_metrics.json`,
`training_history.csv` and `predictions_test.jsonl`, then computes the
dev-tuned threshold test F1.

In [1]:
import json
import os
import pandas as pd

# Point this at the fine-tuned result dir (per model).
RUN_DIR = "out_experiments/m2_beto_es2/mrm8488__bert-base-spanish-wwm-cased-finetuned-spa-squad2-es/merged/ft"

metrics = json.load(open(os.path.join(RUN_DIR, "metrics_summary.json")))
gold = json.load(open(os.path.join(RUN_DIR, "gold_audit_metrics.json")))
history = pd.read_csv(os.path.join(RUN_DIR, "training_history.csv"))

## 1. Dev-tuned threshold (freeze from dev, not test)

Find the best epoch on dev: the epoch with the highest `eval_f1`. Use that
epoch's `eval_best_f1_thresh` as the frozen scalar threshold.

In [2]:
# Best dev epoch = max eval_f1
best_idx = history["eval_f1"].idxmax()
best_epoch = history.loc[best_idx, "epoch"]
best_dev_f1 = history.loc[best_idx, "eval_f1"]
dev_threshold = history.loc[best_idx, "eval_best_f1_thresh"]

print(f"Best dev epoch        : {best_epoch}")
print(f"Best dev eval_f1      : {best_dev_f1:.4f}")
print(f"Frozen dev threshold  : {dev_threshold:.4f}")
print(f"(apply this to test, never the test oracle threshold)")

Best dev epoch        : 3.0
Best dev eval_f1      : 0.7270
Frozen dev threshold  : 2.2017
(apply this to test, never the test oracle threshold)


## 2. Apply the dev threshold to test predictions

In [5]:
import sys
sys.path.insert(0, "scripts")
import qa_dataset_utils as q
from tqdm import tqdm

preds = [json.loads(l) for l in open(os.path.join(RUN_DIR, "predictions_test.jsonl"))]

# Rebuild references from the HF dataset test split (same split used at train time).
from datasets import load_dataset
ds = load_dataset("LeninGF/question-answering-robbery-m2")
ref_map = {r["id"]: r for r in ds["test"]}
refs = []
for p in tqdm(preds):
    r = ref_map[p["id"]]
    refs.append({
        "id": p["id"],
        "is_impossible": bool(r["is_impossible"]),
        "answers": r["answers"],
    })

# Plain (threshold 0.0) on test -> lower bound
plain = q.squad_v2_metrics(preds, refs, threshold=0.0, compute_best=False)

# Dev-tuned threshold applied to test -> headline
headline = q.squad_v2_metrics(preds, refs, threshold=float(dev_threshold), compute_best=False)

# Oracle on test (diagnostic ONLY, do not report as result)
oracle = q.squad_v2_metrics(preds, refs, compute_best=True)

print(f"Downgraded predictions at threshold {dev_threshold:.4f}: "
      f"{len(preds) - headline['total']} unchanged is not applicable (metric recomputed).")
print()
print(f"{'setting':<28} {'EM':>6} {'F1':>6} {'HasAnsF1':>9} {'NoAnsF1':>8}")
def row(name, m):
    print(f"{name:<28} {m['exact']:.4f} {m['f1']:.4f} {m['HasAns_f1']:.4f} {m['NoAns_f1']:.4f}")
row("Test plain (thr 0.0)", plain)
row("Test dev-tuned (HEADLINE)", headline)
row("Test oracle (DIAGNO) ", oracle)

100%|██████████| 8109/8109 [00:00<00:00, 268749.64it/s]


Downgraded predictions at threshold 2.2017: 0 unchanged is not applicable (metric recomputed).

setting                          EM     F1  HasAnsF1  NoAnsF1
Test plain (thr 0.0)         0.5586 0.7274 0.6939 0.8191
Test dev-tuned (HEADLINE)    0.5572 0.7489 0.7599 0.7187
Test oracle (DIAGNO)         0.5586 0.7274 0.6939 0.8191


## 3. Gold audit numbers (report as secondary evaluation)

In [6]:
print("Gold audit (200 rows, threshold 0.0):")
print(f"  EM={gold['exact']:.4f} F1={gold['f1']:.4f} "
      f"HasAnsF1={gold['HasAns_f1']:.4f} NoAnsF1={gold['NoAns_f1']:.4f}")
print()
print("Report plain gold F1 (threshold 0.0); oracle best_f1 on gold is diagnostic only.")

Gold audit (200 rows, threshold 0.0):
  EM=0.3900 F1=0.6074 HasAnsF1=0.6089 NoAnsF1=0.6000

Report plain gold F1 (threshold 0.0); oracle best_f1 on gold is diagnostic only.


## 4. Per-question-type error localization

The weak categories are `place` and `objects` — report these to point where
extractive QA fails on the M2 benchmark.

In [7]:
qtype = pd.read_csv(os.path.join(RUN_DIR, "metrics_by_question_type.csv"))
print(qtype[["kind", "f1", "HasAns_f1", "NoAns_f1"]])
print()
print("Weakest: " + qtype.loc[qtype["f1"].idxmin(), "kind"] +
      f" (f1={qtype['f1'].min():.3f})")

      kind        f1  HasAns_f1  NoAns_f1
0     date  0.896068   0.901714  0.807339
1     time  0.735152   0.731787  0.760638
2    place  0.525959   0.533680  0.496296
3  objects  0.556323   0.538334  0.606796
4    value  0.835837   0.587336  0.966772

Weakest: place (f1=0.526)


## 5. What to write in the paper (summary)

- Report **dev-tuned-threshold test F1** as the headline (a reproducible,
  leakage-free number), plus plain threshold-0 F1 as lower bound.
- Report HasAns / NoAns breakdown (the task includes ~27% unanswerable).
- Report per-question-type EM/F1 and highlight `place` / `objects` as the
  failure modes.
- Report the 200-row gold audit as a second, fully held-out evaluation.
- Do **not** report `best_f1` computed on test (oracle).
- Note the dataset/task difference vs the original paper (all-answerable,
  test-selected, inflated 82.88) so M2 numbers are not compared 1:1.